In [4]:
# 1. Imports
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import sys
import os

# Ensure src modules can be imported (reload so kernel picks up config.py edits)
from importlib import reload
from dotenv import load_dotenv

_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
load_dotenv(os.path.join(_PROJECT_ROOT, '.env'))

from src import config
reload(config)
print(f"Using Groq model: {config.LLM_MODEL}")

# 2. Setup Embeddings & Connect to Vector DB
print("Connecting to local Vector DB...")
embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = Chroma(persist_directory=f"../{config.CHROMA_PATH}", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Setup LLM
print("Initializing Groq LLM...")
llm = ChatGroq(model=config.LLM_MODEL, temperature=0)

# 4. Create the RAG Prompt
prompt_template = """
You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Keep the answer concise.

Question: {question} 
Context: {context} 
Answer:
"""
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def print_retrieved_chunks(docs, question):
    print(f"\n{'=' * 60}")
    print(f"Retrieved {len(docs)} chunk(s) for: {question!r}")
    print("=" * 60)
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        preview = doc.page_content.strip().replace("\n", " ")
        if len(preview) > 300:
            preview = preview[:300] + "..."
        print(f"\n--- Chunk {i} (page {page}, source: {source}) ---")
        print(preview)
    print()

# 5. Create the Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Test the Naive RAG
question = "What is Self-RAG and how does it differ from standard RAG?"
print(f"\nQuestion: {question}")

retrieved_docs = retriever.invoke(question)
print_retrieved_chunks(retrieved_docs, question)

print("Generating Answer...\n")
answer = rag_chain.invoke(question)
print(answer)

Using Groq model: llama-3.3-70b-versatile
Connecting to local Vector DB...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing Groq LLM...

Question: What is Self-RAG and how does it differ from standard RAG?

Generating Answer...

Self-RAG differs from standard RAG in that it concurrently processes multiple retrieved passages, evaluates their relevance, generates task outputs, and critiques its own output to choose the best one. In contrast, standard RAG retrieves a fixed number of documents regardless of necessity and doesn't reassess generation quality.
